# 04 · Level 5: the loop runs itself

`sia.loop.cycle()` runs every step from notebooks 02–03 unattended, as one MLflow run per cycle:

1. **Signal:** read new production traces and compute the failure rate.
2. **Verifier health:** check the judge still agrees with people; halt and ask a person if not.
3. **Store:** add labeled failures to the dataset.
4. **Update:** GEPA, the analyst, a skill, or OpenClaw editing itself.
5. **Gate:** the same rules, plus a bloat guard.
6. **Ship:** `@production` moves, with a receipt for the rollback watch.

This notebook uses its **own** MLflow database (`loop.db`), so production starts from the flawed v1 again.

In [ ]:
from sia import notebook as nb

nb.setup(backend="offline", db="loop.db", fresh=True, experiment="acme-loop-notebook", llm=False)

In [ ]:
from sia import traffic
from sia.agent import pin
from sia.loop import LoopConfig, cycle

nb.ensure_production()
cfg = LoopConfig(improver="analyst")          # llm=False: the rule-based analyst (set llm=True for Claude Opus 5.5)
print(cfg)

Each cycle:
* simulated customers send 16 questions;
* simulated reviewers label half of them;
* `cycle()` decides what to do.

Drift questions (escalations, discounts) start arriving in cycle 3.

In [ ]:
history = []
for i in range(1, 5):
    splits = ("traffic", "drift") if i >= 3 else ("traffic",)
    ids = traffic.send(splits, n=16, seed=i)
    labeled = traffic.label(ids, rate=0.5, seed=i)
    outcome = cycle(cfg)
    history.append({"cycle": i, "traffic": "+".join(splits), "labeled": labeled,
                    "outcome": outcome, "production": pin("production").label})
    print(f"cycle {i}: {outcome}")

In [ ]:
import pandas as pd
pd.DataFrame(history)

## The loop's own history is in MLflow

Every cycle is a run tagged `sia.cycle=true`, with its decision in `sia.outcome` and metrics such as the live
failure rate and dataset size.

In [ ]:
import mlflow
runs = mlflow.search_runs(filter_string="tags.`sia.cycle` = 'true'", order_by=["start_time ASC"])
cols = ["tags.sia.outcome", "tags.sia.candidate", "metrics.live_failure_rate", "metrics.judged_traces", "metrics.dataset_size"]
runs[[c for c in cols if c in runs.columns]]

## What changed, version by version

In [ ]:
from sia import config
from sia.agent import skill_names

client = mlflow.MlflowClient()
for v in sorted(client.search_prompt_versions(config.PROMPT_NAME), key=lambda v: int(v.version)):
    print(f"AGENTS.md v{v.version}  author={v.tags.get('author')}  {v.commit_message[:90]}")
for name in skill_names():
    for v in client.search_prompt_versions(f"{config.SKILL_PREFIX}{name}"):
        print(f"skill {name} v{v.version}  workshop={v.tags.get('workshop_proposal')}  {v.commit_message[:70]}")
print("\nproduction:", pin("production").label)

In [ ]:
# The receipt the rollback watch keeps for the last loop promotion
import json
exp = mlflow.get_experiment_by_name(config.EXPERIMENT)
print(json.dumps(json.loads(exp.tags.get("sia.last_loop_promotion", "null")), indent=2))

## Turning up the dial

* `LoopConfig(improver="gepa" | "analyst" | "skill" | "openclaw", discover=True)` swaps in real improvers and
  MLflow issue discovery. These need a model: set `SIA_PROVIDER` / start Ollama.
* `nb.setup(backend="openclaw")` or `"hermes"` runs the same loop on a real agent framework. Budget 1–3 minutes
  per answer on a local model, so use fewer questions per cycle.
* `python levels/level5_autonomous/02_run_loop.py --interval 1800` keeps it running.